## Load things

In [1]:
%run "/Users/audreyburggraf/Desktop/QUEEN'S/THESIS RESEARCH/PLOTTING C29 989/constants.py"

/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [2]:
%run "/Users/audreyburggraf/Desktop/QUEEN'S/THESIS RESEARCH/PLOTTING C29 989/FUNCTIONS/load_functions"

fortran mie routines unavailable


/opt/anaconda3/lib/python3.9/site-packages/dsharp_opac/dsharp_opac.py:47: UserWarning: could not import compiled mie code - mie calculation will be slow
  warnings.warn(


In [3]:
%matplotlib inline

## Intro stuff

In [4]:
# Constants
bands = [4, 6, 7]
num_bands = len(bands)

lambda_bands_cm = mm_to_cm([lambda_mm[b] for b in bands])
band_colors = [alma_band_colors[b] for b in bands]

In [5]:
# POLF_avg = []

# for band in bands:
#     path = globals()[f"band{band}_data_folder_path"]
    
#     df = pd.read_csv(path + f"constants_BAND{band}.csv")
#     POLF_avg.append(df["POLF_avg"].values[0] * 100)

KeyError: 'POLF_avg'

The Zhang paper axis ticks:

$\log_{10}(a_{max} / cm) = -3 \longrightarrow (a_{max}f) = 10^{-3}$


In [6]:
# Set up wavelength array
logwave_vals = np.linspace(0.1, 4, 10000)

lambda_dist_micron = 10**logwave_vals

lambda_dist_cm = micron_to_cm(lambda_dist_micron)

## Getting data started

In [7]:
f_list = [0.99999, 0.3, 0.1, 0.01]

f_labels = [1, 0.3, 0.1, 0.01]

In [8]:
N_grains = 200

In [9]:
a_max_dist_micron_array = []
a_max_dist_cm_array = []

for i in range(len(f_list)):
    
    print(rf"f = {f_list[i]}")
    
    
    a_max_dist_micron = 1/f_list[i] * np.logspace(np.log10(1e0), np.log10(1e4), N_grains)
    
    a_max_dist_cm = micron_to_cm(a_max_dist_micron)
    
    
    af = a_max_dist_micron*f_list[i]
    log_10_af = np.log10(af)
    
    
    
    a_max_dist_micron_array.append(a_max_dist_micron)
    a_max_dist_cm_array.append(a_max_dist_cm)
    
    print(rf"The minimum a value is: {np.min(a_max_dist_micron)} microns")
    print(rf"The maximum a value is: {np.max(a_max_dist_micron)} microns")
    
    print(rf"The minimum log_10(a*f) value is: {np.min(log_10_af):.2f}")
    print(rf"The maximum log_10(a*f) value is: {np.max(log_10_af):.2f}")
    
    print(" ")
    
    

f = 0.99999
The minimum a value is: 1.000010000100001 microns
The maximum a value is: 10000.10000100001 microns
The minimum log_10(a*f) value is: 0.00
The maximum log_10(a*f) value is: 4.00
 
f = 0.3
The minimum a value is: 3.3333333333333335 microns
The maximum a value is: 33333.333333333336 microns
The minimum log_10(a*f) value is: 0.00
The maximum log_10(a*f) value is: 4.00
 
f = 0.1
The minimum a value is: 10.0 microns
The maximum a value is: 100000.0 microns
The minimum log_10(a*f) value is: 0.00
The maximum log_10(a*f) value is: 4.00
 
f = 0.01
The minimum a value is: 100.0 microns
The maximum a value is: 1000000.0 microns
The minimum log_10(a*f) value is: 0.00
The maximum log_10(a*f) value is: 4.00
 


Reminder: 
P = porosity
f = filling factor 
f = 1 - P or P = 1 - f

In [10]:
# loop

P_times_omega_list = []

for i in range(len(f_list)):
    
    print(rf"Now working on f = {f_list[i]}")
    
    oc, rho_g_cm3 = do.get_dsharp_mix(porosity = 1 - f_list[i])
    
#     res = do.get_opacities(a_max_cm, lambda_dist_cm, rho_g_cm3, oc)
    
    
    mass_grams_fig = calculate_grain_mass(a_max_dist_cm_array[i], rho_g_cm3, "cm")


    res_scatter = do.get_opacities(a_max_dist_cm_array[i], lambda_bands_cm, rho_g_cm3, oc, n_angle=100)

    zscat = do.calculate_mueller_matrix(lambda_bands_cm, mass_grams_fig, 
                                        res_scatter['S1'], res_scatter['S2'], 
                                        theta=res_scatter['theta'], 
                                        k_sca=res_scatter['k_sca'])['zscat']
    theta = res_scatter['theta']


    
    P, omega, P_times_omega = compute_P_and_omega_vs_amax(lambda_bands_cm, a_max_dist_cm_array[i], zscat, res_scatter, theta)
    
    
    P_times_omega_list.append(P_times_omega)

Now working on f = 0.99999
Please cite Warren & Brandt (2008) when using these optical constants
Please cite Draine 2003 when using these optical constants
Reading opacities from troilitek
Please cite Henning & Stognienko (1996) when using these optical constants
Reading opacities from organicsk
Please cite Henning & Stognienko (1996) when using these optical constants
| material                            | volume fractions | mass fractions |
|-------------------------------------|------------------|----------------|
| Water Ice (Warren & Brandt 2008)    | 0.3642           | 0.2            |
| Astronomical Silicates (Draine 2003)| 0.167            | 0.329          |
| Troilite (Henning)                  | 0.02578          | 0.07434        |
| Organics (Henning)                  | 0.443            | 0.3966         |
using Maxwell-Garnett mixing: first component should be host material (= matrix)
    matrix = Vacuum
Mie ... Done!
Now working on f = 0.3
Please cite Warren & Brandt (2008)

/opt/anaconda3/lib/python3.9/site-packages/dsharp_opac/dsharp_opac.py:2116: UserWarning: Maximum error of 9.6%: above error tolerance
  warnings.warn(
/opt/anaconda3/lib/python3.9/site-packages/dsharp_opac/dsharp_opac.py:2116: UserWarning: Maximum error of 88%: above error tolerance
  warnings.warn(


Mie ... Done!
Now working on f = 0.01
Please cite Warren & Brandt (2008) when using these optical constants
Please cite Draine 2003 when using these optical constants
Reading opacities from troilitek
Please cite Henning & Stognienko (1996) when using these optical constants
Reading opacities from organicsk
Please cite Henning & Stognienko (1996) when using these optical constants
| material                            | volume fractions | mass fractions |
|-------------------------------------|------------------|----------------|
| Water Ice (Warren & Brandt 2008)    | 0.3642           | 0.2            |
| Astronomical Silicates (Draine 2003)| 0.167            | 0.329          |
| Troilite (Henning)                  | 0.02578          | 0.07434        |
| Organics (Henning)                  | 0.443            | 0.3966         |
using Maxwell-Garnett mixing: first component should be host material (= matrix)
    matrix = Vacuum
Mie ... 33 %

/opt/anaconda3/lib/python3.9/site-packages/dsharp_opac/dsharp_opac.py:2116: UserWarning: Maximum error of 1.4e+03%: above error tolerance
  warnings.warn(


Mie ... Done!


/opt/anaconda3/lib/python3.9/site-packages/dsharp_opac/dsharp_opac.py:2116: UserWarning: Maximum error of 1.4e+05%: above error tolerance
  warnings.warn(


In [11]:
P_times_omega_array = np.array(P_times_omega_list)

In [13]:
np.shape(P_times_omega_list)

(4, 200, 3)

## Plotting

In [ ]:
# Create a figure with the WCS projection
fig, ax = plt.subplots(1, 4, figsize=(14, 3))



for i in range(4):  # note range(4) since you have 4 subplots
    if i == 0:
        ax[i].set_ylabel('P $\omega$', fontsize=axis_label_fs)
    else:
        ax[i].set_yticklabels([])    # remove y tick labels
        ax[i].set_ylabel('')         # remove y axis label if any
        ax[i].tick_params(left=False)  # optionally remove left ticks
        
    
    # Add axis labels
    ax[i].set_xlabel(r'$\log_{10}(a_{\mathrm{max}}\, f / \mu\mathrm{m})$', fontsize=axis_label_fs)

    
    ax[i].minorticks_on()

    ax[i].tick_params(axis="x", which="minor", direction="in", left=True, right=True, length=4)
    ax[i].tick_params(axis="y", which="minor", direction="in", left=True, right=True, length=4)

    ax[i].tick_params(axis="x", which="major", direction="in", bottom=True, top=True,   length=7, labelsize=axis_num_fs)
    ax[i].tick_params(axis="y", which="major", direction="in", left=True,   right=True, length=7, labelsize=axis_num_fs)


    ax[i].set_title(f'$f$ = {f_list[i]:.2f}', fontsize = 20)


    for j in range(num_bands):
        ax[i].plot(np.log10(a_max_dist_micron_array[i] * f_list[i]),
                P_times_omega_array[i, :, j],
                color = band_colors[j],
                ls = '-',
                label = f'Band {bands[j]}')
        
        ax[i].axhline(POLF_avg[j], color = band_colors[j], ls = '--', lw = 3)
        
    ax[i].set_xlim(0, 4)




# # Remove scientific notation from x-axis
# ax.xaxis.set_major_formatter(ScalarFormatter(useMathText=False))
# ax.ticklabel_format(style='plain', axis='x')

# # ---------------------------------------------------------------------------------
# ax.legend(fontsize = legend_text_fs)
# # ---------------------------------------------------------------------------------
 
    

# ax.set_ylim(-0.1, 1.5)



    
plt.tight_layout()
plt.subplots_adjust(wspace = 0.01, hspace=0)
plt.show()